# Ensemble: V2S + B0 + B3 + SWA-B0 + B0-MIL + SwinMIL

Estratégia final baseada nos resultados do `EXPERIMENTS_REPORT.md`.

**Modelos no ensemble (6 total):**

| # | Modelo | Tipo | Justificativa |
|---|--------|------|---------------|
| 1 | `EfficientNetV2-S + Optuna` | Patch-level | Melhor modelo individual (QWK 0.8742, F1 0.6772) |
| 2 | `EfficientNet-B0` | Patch-level | Melhor custo-benefício; base do ensemble histórico vencedor |
| 3 | `EfficientNet-B3` | Patch-level | Backbone maior; estava no ensemble (B0+B3+B7) que atingiu 0.8752 |
| 4 | `EfficientNet-B0 + SWA` | Patch-level | Stochastic Weight Averaging → trajetória de otimização diferente, erros distintos |
| 5 | `EfficientNetMIL (B0)` | MIL | Captura contexto global complementar (α=0.40 ótimo nos experimentos anteriores) |
| 6 | `SwinMIL` | MIL | Transformer hierárquico para MIL — diversidade arquitetural no raciocínio global |

**Excluídos (com justificativa):**
- `ViT-0`: val QWK ≈ 0.65 → muito fraco, diluiria o ensemble
- `EfficientNet-B7`: overfitting severo (test QWK 0.808 vs val 0.840)
- `B0-HED`: preprocessing customizado não reproduzível com segurança na inferência
- `ConvNeXt`: colapso de treinamento (QWK ~0.55)

**Estratégias avaliadas:** Mean, Weighted-Mean, Geometric Mean, Confidence-Weighted,
Temperature Scaling, Rank Average, Majority Vote, Trimmed Mean,
Alpha search MIL (valida peso ótimo MIL vs CNN), TTA sobre o melhor ensemble.


## 1. Imports & Setup

In [ ]:
import os
import gc
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import DataLoader, SequentialSampler
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, accuracy_score,
    f1_score, recall_score, precision_score,
)
import albumentations as Albu
from albumentations.pytorch import ToTensorV2

sys.path.append('../..')
from utils.dataset import PandasDataset
from utils.mil import PandasWithMilDataset, EfficientNetMIL
from utils.models import EfficientNetApi
from utils.metrics import calculate_metrics, format_metrics

print('Imports OK')

## 2. Configuração

In [ ]:
SEED            = 42
NUM_WORKERS     = 4
OUTPUT_CLASSES  = 5      # ordinal thresholds para ISUP 0-5
BATCH_PATCH     = 4      # batch para modelos patch-level
BATCH_MIL       = 8      # batch para MIL (bags)
MAX_PATCHES     = 36
N_BOOT          = 1000   # iterações bootstrap

# Dropouts dos checkpoints (conforme treinados)
DROPOUT_B0      = 0.6
DROPOUT_V2S     = 0.4422  # valor Optuna
DROPOUT_B3      = 0.6
DROPOUT_SWA     = 0.6
DROPOUT_MIL_B0  = 0.4
DROPOUT_SWMIL   = 0.4
UNFREEZE_V2S    = 3       # blocos descongelados V2-S (Optuna)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

# ── Caminhos ──────────────────────────────────────────────────────────
ROOT_DIR    = '../..'
DATA_DIR    = '../../..'
IMAGES_DIR  = os.path.join(DATA_DIR, 'tiles')
PATCHES_DIR = '/home/woshington/Projects/Doutorado/bag_of_patches'

CKPT_B0    = '../tests/baseline/models/b0-entropy-ordinal-3-focal.pth'
CKPT_V2S   = '../tests/baseline/models/v2-optuna-ordinal-focal.pth'
CKPT_B3    = '../tests/baseline/models/b3-entropy-ordinal-focal.pth'
CKPT_SWA   = '../tests/baseline/models/b0-entropy-ordinal-swa.pth'
CKPT_MIL   = '../tests/transformers/models/b0-mil-focal.pth'
CKPT_SWMIL = '../tests/transformers/models/swin-t-mil-optuna.pth'

os.makedirs('logs', exist_ok=True)

print(f'Device : {DEVICE}')
ckpts = [('B0', CKPT_B0), ('V2S', CKPT_V2S), ('B3', CKPT_B3),
          ('SWA', CKPT_SWA), ('MIL-B0', CKPT_MIL), ('SwinMIL', CKPT_SWMIL)]
for label, path in ckpts:
    status = '✓' if os.path.isfile(path) else '✗ NOT FOUND'
    print(f'  {label:<8}: {path}  [{status}]')


## 3. Definição de Modelos

In [ ]:
# EfficientNetV2Api — wrapper local (mesmo código do notebook de treino)
class EfficientNetV2Api(nn.Module):
    """
    EfficientNetV2-S wrapper com descongelamento parcial de blocos.
    Igual ao usado no treino (efficientnet-v2-optuna-focal.ipynb).
    """
    def __init__(self, model: nn.Module, output_dimensions: int,
                 dropout_rate: float = 0.4, unfreeze_blocks: int = 2):
        super().__init__()
        self.model = model

        for param in self.model.parameters():
            param.requires_grad = False

        if hasattr(self.model, 'features') and unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features

        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, output_dimensions),
        )

    def extract(self, x):
        x = self.model(x)
        if x.ndim == 4:
            x = x.mean(dim=[2, 3])
        return x

    def forward(self, x):
        return self.head(self.extract(x))

## 4. Helpers: Decode, Estratégias de Ensemble

In [ ]:
def decode_ordinal(probs: np.ndarray) -> np.ndarray:
    """Sigmoid probs (N, 5) → classe ISUP 0-5."""
    return (probs > 0.5).sum(axis=1)


def binary_entropy(p: np.ndarray) -> np.ndarray:
    """Entropia binária elemento-a-elemento H(p) = -p*log(p) - (1-p)*log(1-p)."""
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def ensemble_strategies(probs_list: list,
                         weights=None) -> dict:
    """
    Aplica várias estratégias de ensemble sobre uma lista de matrizes de probabilidade.

    Args:
        probs_list: lista de arrays (N, C) — probabilidades sigmoidais de cada modelo
        weights: pesos para a média ponderada (normaliza internamente)

    Returns:
        dict nome → array de predições (N,)
    """
    P = np.stack(probs_list, axis=0)  # (M, N, C)
    M, N, C = P.shape

    results = {}

    # ── 1. Média simples ──────────────────────────────────────────────
    results['Mean'] = decode_ordinal(P.mean(axis=0))

    # ── 2. Média ponderada (QWK-calibrado) ───────────────────────────
    if weights is not None:
        w = np.array(weights, dtype=float)
        w /= w.sum()
        results['Weighted-Mean'] = decode_ordinal(
            (P * w[:, None, None]).sum(axis=0)
        )

    # ── 3. Média geométrica ───────────────────────────────────────────
    log_p = np.log(np.clip(P, 1e-7, 1 - 1e-7))
    results['Geometric-Mean'] = decode_ordinal(np.exp(log_p.mean(axis=0)))

    # ── 4. Mediana ────────────────────────────────────────────────────
    results['Median'] = decode_ordinal(np.median(P, axis=0))

    # ── 5. Trimmed Mean (remove min e max por modelo) ─────────────────
    if M >= 3:
        sorted_p = np.sort(P, axis=0)  # (M, N, C)
        results['Trimmed-Mean'] = decode_ordinal(sorted_p[1:-1].mean(axis=0))
    else:
        results['Trimmed-Mean'] = results['Mean']

    # ── 6. Confidence-Weighted (entropia inversa por amostra) ─────────
    H = binary_entropy(P).mean(axis=2)           # (M, N) — entropia média por threshold
    conf = 1.0 - H / np.log(2)                   # normaliza [0, 1]
    conf = np.clip(conf, 1e-7, None)
    w_conf = conf / conf.sum(axis=0, keepdims=True)  # (M, N)
    results['Conf-Weighted'] = decode_ordinal(
        (P * w_conf[:, :, None]).sum(axis=0)
    )

    # ── 7. Temperature Scaling T=0.5 (aguça distribuição) ────────────
    logits = np.log(np.clip(P, 1e-7, 1 - 1e-7)) - np.log(np.clip(1 - P, 1e-7, 1 - 1e-7))
    results['Temp-0.5'] = decode_ordinal(
        1 / (1 + np.exp(-(logits / 0.5).mean(axis=0)))
    )

    # ── 8. Temperature Scaling T=2.0 (suaviza distribuição) ──────────
    results['Temp-2.0'] = decode_ordinal(
        1 / (1 + np.exp(-(logits / 2.0).mean(axis=0)))
    )

    # ── 9. Rank Average (média de grades decodificadas) ───────────────
    grades = np.stack([decode_ordinal(p) for p in probs_list], axis=0).astype(float)  # (M, N)
    results['Rank-Avg'] = np.clip(np.round(grades.mean(axis=0)), 0, 5).astype(int)

    # ── 10. Majority Vote (voto majoritário hard) ─────────────────────
    from scipy.stats import mode
    results['Majority-Vote'] = mode(grades.astype(int), axis=0, keepdims=False).mode

    return results


def compute_metrics_bootstrap(targets: np.ndarray, preds: np.ndarray,
                               n_boot: int = 1000, seed: int = 42) -> dict:
    """Bootstrap 95% CI para accuracy, QWK, F1-macro, recall, precision."""
    rng = np.random.default_rng(seed)
    n = len(targets)
    stats = {k: [] for k in ['acc', 'kappa', 'f1', 'recall', 'precision']}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        t, p = targets[idx], preds[idx]
        stats['acc'].append(accuracy_score(t, p))
        stats['kappa'].append(cohen_kappa_score(t, p, weights='quadratic'))
        stats['f1'].append(f1_score(t, p, average='macro', zero_division=0))
        stats['recall'].append(recall_score(t, p, average='macro', zero_division=0))
        stats['precision'].append(precision_score(t, p, average='macro', zero_division=0))

    def ci(v):
        v = np.array(v)
        return {'mean': v.mean(), 'std': v.std(ddof=1),
                'ci_5': np.percentile(v, 5), 'ci_95': np.percentile(v, 95)}

    return {k: ci(v) for k, v in stats.items()}


def print_metrics(name: str, m: dict):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    for k, v in m.items():
        label = k.upper().replace('_', '-')
        print(f"  {label:<12} {v['mean']*100 if k=='acc' else v['mean']:.4f}"
              f"  ±{v['std']:.4f}  95%CI [{v['ci_5']:.4f}, {v['ci_95']:.4f}]")


def make_tta_transforms():
    """8 geometric TTA views: original + HFlip + VFlip + 3 rotations + Transpose + HFlip+VFlip."""
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    def _build(extra):
        ops = extra + [Albu.Normalize(mean=mean, std=std), ToTensorV2()]
        return Albu.Compose(ops)

    return [
        _build([]),
        _build([Albu.HorizontalFlip(p=1)]),
        _build([Albu.VerticalFlip(p=1)]),
        _build([Albu.Rotate(limit=(90, 90), p=1)]),
        _build([Albu.Rotate(limit=(180, 180), p=1)]),
        _build([Albu.Rotate(limit=(270, 270), p=1)]),
        _build([Albu.Transpose(p=1)]),
        _build([Albu.HorizontalFlip(p=1), Albu.VerticalFlip(p=1)]),
    ]


print('Helpers definidos.')

## 5. Carregamento dos Modelos

In [ ]:
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import swin_t, Swin_T_Weights
from utils.mil import SwinMIL

# ── EfficientNet-B0 ───────────────────────────────────────────────────
print('Carregando EfficientNet-B0...')
backbone_b0 = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model_b0 = EfficientNetApi(model=backbone_b0, output_dimensions=OUTPUT_CLASSES,
                            dropout_rate=DROPOUT_B0)
model_b0.load_state_dict(torch.load(CKPT_B0, weights_only=True))
model_b0 = model_b0.to(DEVICE).eval()
print('  ✓ B0')

# ── EfficientNetV2-S + Optuna ─────────────────────────────────────────
print('Carregando EfficientNetV2-S + Optuna...')
backbone_v2s = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
model_v2s = EfficientNetV2Api(model=backbone_v2s, output_dimensions=OUTPUT_CLASSES,
                               dropout_rate=DROPOUT_V2S, unfreeze_blocks=UNFREEZE_V2S)
model_v2s.load_state_dict(torch.load(CKPT_V2S, weights_only=True))
model_v2s = model_v2s.to(DEVICE).eval()
print('  ✓ V2S')

# ── EfficientNet-B3 ───────────────────────────────────────────────────
print('Carregando EfficientNet-B3...')
backbone_b3 = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
model_b3 = EfficientNetApi(model=backbone_b3, output_dimensions=OUTPUT_CLASSES,
                            dropout_rate=DROPOUT_B3)
model_b3.load_state_dict(torch.load(CKPT_B3, weights_only=True))
model_b3 = model_b3.to(DEVICE).eval()
print('  ✓ B3')

# ── EfficientNet-B0 + SWA ─────────────────────────────────────────────
# SWA: pesos médios de vários checkpoints ao longo do treinamento
print('Carregando EfficientNet-B0 + SWA...')
backbone_swa = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model_swa = EfficientNetApi(model=backbone_swa, output_dimensions=OUTPUT_CLASSES,
                             dropout_rate=DROPOUT_SWA)
model_swa.load_state_dict(torch.load(CKPT_SWA, weights_only=True))
model_swa = model_swa.to(DEVICE).eval()
print('  ✓ SWA')

# ── EfficientNetMIL (B0 backbone) ────────────────────────────────────
print('Carregando EfficientNetMIL (B0)...')
backbone_mil = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model_mil = EfficientNetMIL(
    model=backbone_mil,
    output_classes=OUTPUT_CLASSES,
    dropout_rate=DROPOUT_MIL_B0,
    hidden_dim=512,
    gated=True,
    pool='att',
)
model_mil.load_state_dict(torch.load(CKPT_MIL, weights_only=True))
model_mil = model_mil.to(DEVICE).eval()
print('  ✓ MIL-B0')

# ── SwinMIL (Swin-T backbone + GatedAttention) ────────────────────────
print('Carregando SwinMIL...')
backbone_swmil = swin_t(weights=Swin_T_Weights.DEFAULT)
model_swmil = SwinMIL(
    model=backbone_swmil,
    output_classes=OUTPUT_CLASSES,
    unfreeze_last_blocks=2,
    dropout_rate=DROPOUT_SWMIL,
    hidden_dim=512,
    gated=True,
    pool='att',
)
model_swmil.load_state_dict(torch.load(CKPT_SWMIL, weights_only=True))
model_swmil = model_swmil.to(DEVICE).eval()
print('  ✓ SwinMIL')

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    print(f'\nVRAM alocada: {alloc:.2f} GB')
else:
    print('\nRodando em CPU')


## 6. Carregamento dos Dados

In [ ]:
# ── CSVs ─────────────────────────────────────────────────────────────
df_all     = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_test    = pd.read_csv(f'{ROOT_DIR}/data/test.csv')

df_all.columns     = df_all.columns.str.strip()
df_entropy.columns = df_entropy.columns.str.strip()
df_test.columns    = df_test.columns.str.strip()

# ── Filtragem por entropia (mesmo critério do treino: top 20%) ────────
hard_ids = set(
    df_entropy.sort_values('difficulty_score', ascending=False)
              .head(int(len(df_entropy) * 0.2))['image_id']
)
df_all = df_all[~df_all['image_id'].isin(hard_ids)].reset_index(drop=True)

# Fold 3 → validação (usado para calibrar pesos do ensemble)
df_val = df_all[df_all['fold'] == 3].reset_index(drop=True)

# ── Filtra imagens inexistentes ───────────────────────────────────────
def filter_existing(df: pd.DataFrame, images_dir: str) -> pd.DataFrame:
    exists = df['image_id'].apply(
        lambda x: os.path.isfile(os.path.join(images_dir, f'{x}.png'))
    )
    return df[exists].reset_index(drop=True)

def filter_existing_bags(df: pd.DataFrame, patches_dir: str) -> pd.DataFrame:
    exists = df['image_id'].apply(
        lambda x: os.path.isdir(os.path.join(patches_dir, str(x)))
    )
    return df[exists].reset_index(drop=True)

df_test_patch = filter_existing(df_test, IMAGES_DIR)
df_val_patch  = filter_existing(df_val,  IMAGES_DIR)

df_test_mil   = filter_existing_bags(df_test, PATCHES_DIR)
df_val_mil    = filter_existing_bags(df_val,  PATCHES_DIR)

# Interseção: imagens presentes em AMBOS os formatos
common_test = set(df_test_patch['image_id']) & set(df_test_mil['image_id'])
common_val  = set(df_val_patch['image_id'])  & set(df_val_mil['image_id'])

df_test_common = df_test_patch[df_test_patch['image_id'].isin(common_test)].reset_index(drop=True)
df_val_common  = df_val_patch [df_val_patch['image_id'].isin(common_val)].reset_index(drop=True)

print(f'Test  (common): {len(df_test_common)} amostras')
print(f'Val   (common): {len(df_val_common)} amostras')
print(f'Distribuição ISUP teste:\n{df_test_common["isup_grade"].value_counts().sort_index()}')

## 7. Funções de Inferência

In [ ]:
def get_patch_probs(model, df, images_dir, batch_size, num_workers, device,
                    tta_transforms=None, desc='Patch inference'):
    """
    Roda inferência patch-level. Se tta_transforms for uma lista de Albu.Compose,
    faz TTA e retorna a média das probabilidades.
    Returns: probs (N,C), targets (N,), img_ids list[str]
    """
    all_probs_tta = []
    targets, img_ids = None, None  # coletados apenas no primeiro pass
    transform_list = tta_transforms if tta_transforms else [None]

    for transform in transform_list:
        ds = PandasDataset(images_dir, df, transforms=transform, format='png')
        dl = DataLoader(ds, batch_size=batch_size, num_workers=num_workers,
                        sampler=SequentialSampler(ds), pin_memory=True)
        run_probs, run_targets, run_ids = [], [], []
        model.eval()
        with torch.no_grad():
            for batch_x, batch_y, batch_ids in tqdm(dl, desc=desc, leave=False):
                probs = torch.sigmoid(model(batch_x.to(device))).cpu().numpy()
                run_probs.append(probs)
                if targets is None:  # coleta targets/ids apenas no primeiro transform
                    run_targets.extend(batch_y.sum(1).long().tolist())
                    run_ids.extend([str(x) for x in batch_ids])

        all_probs_tta.append(np.vstack(run_probs))
        if targets is None:
            targets = np.array(run_targets)
            img_ids = run_ids

    return np.mean(all_probs_tta, axis=0), targets, img_ids


def get_mil_probs(model, df, patches_dir, batch_size, num_workers, device,
                  desc='MIL inference'):
    """
    Roda inferência MIL.
    Returns: probs (N,C), targets (N,), img_ids list[str]
    """
    ds = PandasWithMilDataset(patches_dir, df, transforms=None, max_patches=MAX_PATCHES)
    dl = DataLoader(ds, batch_size=batch_size, num_workers=num_workers,
                    sampler=SequentialSampler(ds), pin_memory=True)

    all_probs, all_targets, all_ids = [], [], []
    model.eval()
    with torch.no_grad():
        for bag, mask, tgts, batch_ids in tqdm(dl, desc=desc, leave=False):
            bag  = bag.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            with torch.autocast(device_type=str(device).split(':')[0],
                                 dtype=torch.float16, enabled=torch.cuda.is_available()):
                out = model(bag, mask)
            probs = torch.sigmoid(out['logits']).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(tgts.sum(1).long().numpy())
            all_ids.extend([str(x) for x in batch_ids])

    return np.vstack(all_probs), np.concatenate(all_targets), all_ids


def align_modalities(data_dict):
    """
    Alinha N modalidades por image_id. Retorna apenas amostras presentes em TODAS.

    Args:
        data_dict: {nome: (probs np.ndarray, targets np.ndarray, img_ids list)}

    Returns:
        common_ids (list), targets (np.ndarray), {nome: probs np.ndarray}
    """
    names = list(data_dict.keys())
    frames = {}
    for name, (probs, tgts, ids) in data_dict.items():
        df_m = pd.DataFrame(
            {'target': tgts, 'row_idx': np.arange(len(tgts))},
            index=ids
        )
        df_m = df_m[~df_m.index.duplicated(keep='first')]
        df_m['probs'] = [probs[i] for i in df_m['row_idx']]
        frames[name] = df_m

    common_idx = frames[names[0]].index
    for name in names[1:]:
        common_idx = common_idx.intersection(frames[name].index)

    aligned_probs = {}
    ref_targets = None
    for name in names:
        sub = frames[name].loc[common_idx]
        aligned_probs[name] = np.vstack(sub['probs'].tolist())
        if ref_targets is None:
            ref_targets = sub['target'].values.astype(int)
        else:
            n_diff = (ref_targets != sub['target'].values).sum()
            if n_diff > 0:
                print(f'[AVISO] {n_diff} targets divergem entre {names[0]} e {name} — usando {names[0]}.')

    print(f'Amostras comuns ({" + ".join(names)}): {len(common_idx)}')
    return list(common_idx), ref_targets, aligned_probs


print('Funções de inferência + align_modalities definidas.')


## 8. Coleta de Probabilidades (Teste + Validação)

In [ ]:
print('── Inferência no conjunto de TESTE ──────────────────────────────')
test_probs_b0,   test_tgts_b0,   test_ids_b0   = get_patch_probs(model_b0,   df_test_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='B0   test')
test_probs_v2s,  test_tgts_v2s,  test_ids_v2s  = get_patch_probs(model_v2s,  df_test_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='V2S  test')
test_probs_b3,   test_tgts_b3,   test_ids_b3   = get_patch_probs(model_b3,   df_test_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='B3   test')
test_probs_swa,  test_tgts_swa,  test_ids_swa  = get_patch_probs(model_swa,  df_test_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='SWA  test')
test_probs_mil,  test_tgts_mil,  test_ids_mil  = get_mil_probs  (model_mil,  df_test_common, PATCHES_DIR, BATCH_MIL, NUM_WORKERS, DEVICE, desc='MIL  test')
test_probs_swmil,test_tgts_swmil,test_ids_swmil= get_mil_probs  (model_swmil,df_test_common, PATCHES_DIR, BATCH_MIL, NUM_WORKERS, DEVICE, desc='SwMIL test')

# Alinha todas as modalidades por image_id
test_img_ids, test_targets, test_probs_aligned = align_modalities({
    'b0':    (test_probs_b0,   test_tgts_b0,   test_ids_b0),
    'v2s':   (test_probs_v2s,  test_tgts_v2s,  test_ids_v2s),
    'b3':    (test_probs_b3,   test_tgts_b3,   test_ids_b3),
    'swa':   (test_probs_swa,  test_tgts_swa,  test_ids_swa),
    'mil':   (test_probs_mil,  test_tgts_mil,  test_ids_mil),
    'swmil': (test_probs_swmil,test_tgts_swmil,test_ids_swmil),
})
test_probs_b0    = test_probs_aligned['b0']
test_probs_v2s   = test_probs_aligned['v2s']
test_probs_b3    = test_probs_aligned['b3']
test_probs_swa   = test_probs_aligned['swa']
test_probs_mil   = test_probs_aligned['mil']
test_probs_swmil = test_probs_aligned['swmil']
print(f'  Test: {len(test_targets)} amostras | shape: {test_probs_b0.shape}')

print('── Inferência no conjunto de VALIDAÇÃO (calibração de pesos) ────')
val_probs_b0,   val_tgts_b0,   val_ids_b0   = get_patch_probs(model_b0,   df_val_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='B0   val')
val_probs_v2s,  val_tgts_v2s,  val_ids_v2s  = get_patch_probs(model_v2s,  df_val_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='V2S  val')
val_probs_b3,   val_tgts_b3,   val_ids_b3   = get_patch_probs(model_b3,   df_val_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='B3   val')
val_probs_swa,  val_tgts_swa,  val_ids_swa  = get_patch_probs(model_swa,  df_val_common, IMAGES_DIR, BATCH_PATCH, NUM_WORKERS, DEVICE, desc='SWA  val')
val_probs_mil,  val_tgts_mil,  val_ids_mil  = get_mil_probs  (model_mil,  df_val_common, PATCHES_DIR, BATCH_MIL, NUM_WORKERS, DEVICE, desc='MIL  val')
val_probs_swmil,val_tgts_swmil,val_ids_swmil= get_mil_probs  (model_swmil,df_val_common, PATCHES_DIR, BATCH_MIL, NUM_WORKERS, DEVICE, desc='SwMIL val')

val_img_ids, val_targets, val_probs_aligned = align_modalities({
    'b0':    (val_probs_b0,   val_tgts_b0,   val_ids_b0),
    'v2s':   (val_probs_v2s,  val_tgts_v2s,  val_ids_v2s),
    'b3':    (val_probs_b3,   val_tgts_b3,   val_ids_b3),
    'swa':   (val_probs_swa,  val_tgts_swa,  val_ids_swa),
    'mil':   (val_probs_mil,  val_tgts_mil,  val_ids_mil),
    'swmil': (val_probs_swmil,val_tgts_swmil,val_ids_swmil),
})
val_probs_b0    = val_probs_aligned['b0']
val_probs_v2s   = val_probs_aligned['v2s']
val_probs_b3    = val_probs_aligned['b3']
val_probs_swa   = val_probs_aligned['swa']
val_probs_mil   = val_probs_aligned['mil']
val_probs_swmil = val_probs_aligned['swmil']
print(f'  Val : {len(val_targets)} amostras')


## 9. Calibração de Pesos por Validação

### 9a. Peso QWK individual

Pesos proporcionais ao QWK de cada modelo na validação → usados na Weighted-Mean.

In [ ]:
# QWK individual de cada modelo na validação
val_preds_b0    = decode_ordinal(val_probs_b0)
val_preds_v2s   = decode_ordinal(val_probs_v2s)
val_preds_b3    = decode_ordinal(val_probs_b3)
val_preds_swa   = decode_ordinal(val_probs_swa)
val_preds_mil   = decode_ordinal(val_probs_mil)
val_preds_swmil = decode_ordinal(val_probs_swmil)

model_names = ['B0', 'V2S', 'B3', 'SWA', 'MIL-B0', 'SwinMIL']
val_preds_all = [val_preds_b0, val_preds_v2s, val_preds_b3,
                 val_preds_swa, val_preds_mil, val_preds_swmil]

kappas = np.array([
    cohen_kappa_score(val_targets, p, weights='quadratic')
    for p in val_preds_all
])
qwk_weights = kappas / kappas.sum()

print('QWK individual na validação:')
for name, k, w in zip(model_names, kappas, qwk_weights):
    print(f'  {name:<10}  QWK={k:.4f}  peso={w:.4f}')


### 9b. Alpha search: peso ótimo do MIL vs CNN-ensemble

Busca o α que maximiza QWK na validação para:
`ensemble = α * MIL + (1-α) * mean(B0, V2S)`

In [ ]:
# CNN ensemble = média dos 4 modelos patch-level
alphas     = np.arange(0.0, 1.05, 0.05)
val_kappas = []

# MIL ensemble = média dos 2 modelos MIL
val_cnn_mean = (val_probs_b0 + val_probs_v2s + val_probs_b3 + val_probs_swa) / 4.0
val_mil_mean = (val_probs_mil + val_probs_swmil) / 2.0

for alpha in alphas:
    mixed = alpha * val_mil_mean + (1 - alpha) * val_cnn_mean
    preds = decode_ordinal(mixed)
    k = cohen_kappa_score(val_targets, preds, weights='quadratic')
    val_kappas.append(k)

best_alpha = float(alphas[np.argmax(val_kappas)])
best_val_k = float(np.max(val_kappas))
print(f'Alpha ótimo MIL vs CNN (val): {best_alpha:.2f}  →  QWK={best_val_k:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(alphas, val_kappas, marker='o', linewidth=1.5)
plt.axvline(best_alpha, color='red', linestyle='--', label=f'α={best_alpha:.2f}')
plt.xlabel('α (peso do ensemble MIL)')
plt.ylabel('QWK validação')
plt.title('Alpha Search: MIL-mean vs CNN-mean (B0+V2S+B3+SWA)')
plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout()
plt.savefig('logs/alpha-search-mil-cnn.png', dpi=200)
plt.show()


## 10. Avaliação de Todas as Estratégias (Conjunto de Teste)

In [ ]:
# ── Agregados de base ────────────────────────────────────────────────
cnn_mean_test = (test_probs_b0 + test_probs_v2s + test_probs_b3 + test_probs_swa) / 4.0
mil_mean_test = (test_probs_mil + test_probs_swmil) / 2.0

all_cnn  = [test_probs_b0, test_probs_v2s, test_probs_b3, test_probs_swa]
all_mil  = [test_probs_mil, test_probs_swmil]
all_six  = all_cnn + all_mil

# ── Mapas de probabilidade para comparação ───────────────────────────
probs_map = {
    # Individuais
    'B0':             test_probs_b0,
    'V2S':            test_probs_v2s,
    'B3':             test_probs_b3,
    'SWA':            test_probs_swa,
    'MIL-B0':         test_probs_mil,
    'SwinMIL':        test_probs_swmil,
    # Sub-ensembles CNN
    'CNN-B0+V2S':     (test_probs_b0 + test_probs_v2s) / 2.0,
    'CNN-Mean(4)':    cnn_mean_test,
    # Sub-ensembles MIL
    'MIL-Mean(2)':    mil_mean_test,
    # Combinações CNN + MIL
    f'Alpha-{best_alpha:.2f}': best_alpha * mil_mean_test + (1 - best_alpha) * cnn_mean_test,
    'All6-Mean':      np.mean(all_six, axis=0),
}

# ── Estratégias avançadas sobre os 6 modelos ─────────────────────────
six_strategies = ensemble_strategies(all_six, weights=qwk_weights.tolist())

# ── Métricas bootstrap ───────────────────────────────────────────────
all_results = {}

for name, probs in probs_map.items():
    preds = decode_ordinal(probs)
    all_results[name] = compute_metrics_bootstrap(test_targets, preds, N_BOOT)

for name, preds in six_strategies.items():
    all_results[f'Six-{name}'] = compute_metrics_bootstrap(test_targets, preds, N_BOOT)

print(f'{len(all_results)} configurações avaliadas.')


## 11. Tabela de Resultados

In [ ]:
# ── Montar tabela ordenada por QWK ───────────────────────────────────
rows = []
for name, m in all_results.items():
    rows.append({
        'Modelo / Estratégia': name,
        'Acurácia (%)':  round(m['acc']['mean']   * 100, 2),
        'QWK':           round(m['kappa']['mean'],        4),
        'F1-Macro':      round(m['f1']['mean'],           4),
        'Recall':        round(m['recall']['mean'],       4),
        'Precision':     round(m['precision']['mean'],    4),
        'QWK CI 5%':     round(m['kappa']['ci_5'],        4),
        'QWK CI 95%':    round(m['kappa']['ci_95'],       4),
    })

df_results = pd.DataFrame(rows).sort_values('QWK', ascending=False).reset_index(drop=True)

print('\nRESULTADOS COMPLETOS (ordenado por QWK)\n')
print(df_results.to_string(index=False))

best_name = df_results.iloc[0]['Modelo / Estratégia']
best_kappa = df_results.iloc[0]['QWK']
print(f"\n★ Melhor estratégia: {best_name}  QWK={best_kappa:.4f}")

In [ ]:
# ── Visualização: barras de QWK ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#e74c3c' if i == 0 else '#3498db' if i < 3 else '#95a5a6'
          for i in range(len(df_results))]

bars = ax.barh(df_results['Modelo / Estratégia'][::-1],
               df_results['QWK'][::-1],
               color=colors[::-1], edgecolor='white', height=0.7)

# Barras de erro (CI 5-95)
xerr_low  = (df_results['QWK'] - df_results['QWK CI 5%'])[::-1].values
xerr_high = (df_results['QWK CI 95%'] - df_results['QWK'])[::-1].values
ax.errorbar(df_results['QWK'][::-1].values,
            range(len(df_results)),
            xerr=[xerr_low, xerr_high],
            fmt='none', color='black', capsize=3, linewidth=1)

for bar, v in zip(bars, df_results['QWK'][::-1].values):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)

ax.set_xlabel('Quadratic Weighted Kappa (QWK)', fontsize=11)
ax.set_title('Comparação de Estratégias de Ensemble\nV2S + B0 + MIL', fontsize=12)
ax.axvline(0.87, color='green', linestyle='--', alpha=0.7, label='QWK=0.87 (referência)')
ax.legend()
ax.set_xlim(max(0, df_results['QWK'].min() - 0.02), df_results['QWK'].max() + 0.015)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('logs/ensemble-v2s-b0-mil-comparison.png', dpi=200)
plt.show()

## 12. TTA sobre o Melhor Ensemble

In [ ]:
print('Coletando probabilidades com TTA (8 vistas) nos modelos patch...')
tta_transforms = make_tta_transforms()

test_probs_b0_tta,  *_ = get_patch_probs(model_b0,  df_test_common, IMAGES_DIR,
                                          BATCH_PATCH, NUM_WORKERS, DEVICE,
                                          tta_transforms=tta_transforms, desc='B0  TTA')
test_probs_v2s_tta, *_ = get_patch_probs(model_v2s, df_test_common, IMAGES_DIR,
                                          BATCH_PATCH, NUM_WORKERS, DEVICE,
                                          tta_transforms=tta_transforms, desc='V2S TTA')
test_probs_b3_tta,  *_ = get_patch_probs(model_b3,  df_test_common, IMAGES_DIR,
                                          BATCH_PATCH, NUM_WORKERS, DEVICE,
                                          tta_transforms=tta_transforms, desc='B3  TTA')
test_probs_swa_tta, *_ = get_patch_probs(model_swa, df_test_common, IMAGES_DIR,
                                          BATCH_PATCH, NUM_WORKERS, DEVICE,
                                          tta_transforms=tta_transforms, desc='SWA TTA')
# MIL não tem TTA (bags já são multi-instância, amostragem múltipla não se aplica)
print('TTA coletado para os 4 modelos CNN.')

# Alinha TTA com test_img_ids (mesma interseção)
tta_img_ids, _, tta_probs_aligned = align_modalities({
    'b0':  (test_probs_b0_tta,  test_targets, test_img_ids),
    'v2s': (test_probs_v2s_tta, test_targets, test_img_ids),
    'b3':  (test_probs_b3_tta,  test_targets, test_img_ids),
    'swa': (test_probs_swa_tta, test_targets, test_img_ids),
})

cnn_mean_tta = np.mean([tta_probs_aligned[k] for k in ['b0','v2s','b3','swa']], axis=0)
alpha_tta    = best_alpha * mil_mean_test + (1 - best_alpha) * cnn_mean_tta

preds_noTTA = decode_ordinal(best_alpha * mil_mean_test + (1 - best_alpha) * cnn_mean_test)
preds_TTA   = decode_ordinal(alpha_tta)

metrics_noTTA = compute_metrics_bootstrap(test_targets, preds_noTTA, N_BOOT)
metrics_TTA   = compute_metrics_bootstrap(test_targets, preds_TTA,   N_BOOT)

print(f"Sem TTA — QWK: {metrics_noTTA['kappa']['mean']:.4f}  Acc: {metrics_noTTA['acc']['mean']*100:.2f}%")
print(f"Com TTA — QWK: {metrics_TTA['kappa']['mean']:.4f}  Acc: {metrics_TTA['acc']['mean']*100:.2f}%")
print(f"Delta QWK TTA: {metrics_TTA['kappa']['mean'] - metrics_noTTA['kappa']['mean']:+.4f}")

all_results[f'Alpha-{best_alpha:.2f}+TTA'] = metrics_TTA

## 13. Melhor Modelo — Análise Detalhada

In [ ]:
# Re-rankeia incluindo TTA
rows_final = []
for name, m in all_results.items():
    rows_final.append({
        'Modelo / Estratégia': name,
        'Acurácia (%)': round(m['acc']['mean']   * 100, 2),
        'QWK':          round(m['kappa']['mean'],        4),
        'F1-Macro':     round(m['f1']['mean'],           4),
    })

df_final = pd.DataFrame(rows_final).sort_values('QWK', ascending=False).reset_index(drop=True)
best_name_final = df_final.iloc[0]['Modelo / Estratégia']
print(f"Melhor configuração final: {best_name_final}")

# Determina as predições do melhor modelo
if 'TTA' in best_name_final:
    best_preds = preds_TTA
elif best_name_final in probs_map:
    best_preds = decode_ordinal(probs_map[best_name_final])
elif best_name_final.startswith('Six-'):
    strat_key = best_name_final.replace('Six-', '')
    best_preds = six_strategies.get(strat_key, decode_ordinal(np.mean(all_six, axis=0)))
else:
    best_preds = decode_ordinal(
        best_alpha * mil_mean_test + (1 - best_alpha) * cnn_mean_test
    )

# Classification report
labels = [f'ISUP {i}' for i in range(6)]
print(f'\nClassification Report — {best_name_final}\n')
print(classification_report(test_targets, best_preds, target_names=labels, digits=4, zero_division=0))

In [ ]:
# Matrizes de confusão (contagens + normalizada)
cm      = confusion_matrix(test_targets, best_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix (contagens)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalizada)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted')

plt.suptitle(f'Ensemble: {best_name_final}', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('logs/ensemble-v2s-b0-mil-confusion-matrix.png', dpi=200)
plt.show()

## 14. Comparação com Baseline (EXPERIMENTS_REPORT.md)

In [ ]:
# Referências históricas do relatório
baseline_reference = {
    'Specialist-Base (B0+B3+B7)': {'qwk': 0.8754, 'acc': 71.36},
    'Baseline-Mean (B0+B3+B7)':   {'qwk': 0.8752, 'acc': 71.26},
    'V2S+Optuna (individual)':    {'qwk': 0.8742, 'acc': 71.51},
    'Ensemble+TTA (B0+B3+B7)':    {'qwk': 0.8750, 'acc': 71.40},
    'MIL+Patch α=0.40 (B0)':      {'qwk': 0.8670, 'acc': 70.50},
}

print('\nComparação com resultados anteriores (EXPERIMENTS_REPORT.md)\n')
print(f'{"Modelo":<45} {"QWK":>8} {"Acc":>8}')
print('-' * 65)

for name, ref in baseline_reference.items():
    print(f'{name:<45} {ref["qwk"]:>8.4f} {ref["acc"]:>7.2f}%')

print('\nNovos resultados:')
for _, row in df_final.head(5).iterrows():
    name = row['Modelo / Estratégia']
    k    = row['QWK']
    acc  = row['Acurácia (%)']
    best_prev = 0.8754
    delta = k - best_prev
    marker = '★ NOVO MELHOR' if delta > 0 else f'Δ={delta:+.4f}'
    print(f'{name:<45} {k:>8.4f} {acc:>7.2f}%  {marker}')

## 15. Salvar Resultados

In [ ]:
# ── CSV com predições de todos os modelos ───────────────────────────
oracle_df = pd.DataFrame({
    'image_id':    test_img_ids,
    'true_label':  test_targets,
    'pred_b0':     decode_ordinal(test_probs_b0),
    'pred_v2s':    decode_ordinal(test_probs_v2s),
    'pred_b3':     decode_ordinal(test_probs_b3),
    'pred_swa':    decode_ordinal(test_probs_swa),
    'pred_mil_b0': decode_ordinal(test_probs_mil),
    'pred_swmil':  decode_ordinal(test_probs_swmil),
    'pred_best':   best_preds,
})
oracle_df.to_csv('logs/ensemble-v2s-b0-mil-oracle.csv', index=False)

# ── Relatório TXT ─────────────────────────────────────────────────────
report_path = 'logs/ensemble-v2s-b0-mil-results.txt'
with open(report_path, 'w') as f:
    f.write('Ensemble: V2S + B0 + B3 + SWA + B0-MIL + SwinMIL\n')
    f.write('=' * 80 + '\n\n')
    f.write(f'Alpha MIL ótimo (val): {best_alpha:.2f}  QWK_val={best_val_k:.4f}\n')
    f.write('Pesos QWK-calibrados:\n')
    for name, k, w in zip(model_names, kappas, qwk_weights):
        f.write(f'  {name:<10}  QWK_val={k:.4f}  peso={w:.4f}\n')
    f.write(f'\nBootstrap resamples: {N_BOOT}\n\n')
    f.write(f'{"Modelo / Estratégia":<45} {"Accuracy":>10} {"QWK":>10} {"F1-Macro":>10}\n')
    f.write('-' * 80 + '\n')
    for _, row in df_final.iterrows():
        f.write(f'{row["Modelo / Estratégia"]:<45} {row["Acurácia (%)"]:>9.2f}% '
                f'{row["QWK"]:>10.4f} {row["F1-Macro"]:>10.4f}\n')
    f.write('\n' + '=' * 80 + '\n')
    f.write(f'Melhor: {best_name_final}  QWK={df_final.iloc[0]["QWK"]:.4f}\n\n')
    f.write('Classification Report (melhor):\n')
    f.write(classification_report(test_targets, best_preds,
                                   target_names=labels, digits=4, zero_division=0))
    f.write('\nConfusion Matrix:\n' + str(cm) + '\n')
    delta_best = df_final.iloc[0]['QWK'] - 0.8754
    f.write(f'\nDelta vs melhor histórico (0.8754): {delta_best:+.4f}\n')

print(f'Resultados salvos → {report_path}')
print(f'Oracle CSV       → logs/ensemble-v2s-b0-mil-oracle.csv')


## 16. Sumário Final

Exibe um sumário compacto comparando o melhor resultado deste notebook com os benchmarks históricos do `EXPERIMENTS_REPORT.md`.

In [ ]:
best_row = df_final.iloc[0]
best_m   = all_results[best_row['Modelo / Estratégia']]

print('\n' + '=' * 70)
print('  SUMÁRIO FINAL — Ensemble V2S + B0 + MIL')
print('=' * 70)
print(f'  Melhor estratégia : {best_row["Modelo / Estratégia"]}')
print(f'  Acurácia          : {best_row["Acurácia (%)"]:.2f}%')
print(f'  QWK               : {best_row["QWK"]:.4f}  '
      f'95%CI [{best_m["kappa"]["ci_5"]:.4f}, {best_m["kappa"]["ci_95"]:.4f}]')
print(f'  F1-Macro          : {best_row["F1-Macro"]:.4f}  '
      f'95%CI [{best_m["f1"]["ci_5"]:.4f}, {best_m["f1"]["ci_95"]:.4f}]')
print(f'  Alpha MIL ótimo   : {best_alpha:.2f}')
print(f'  TTA incluída      : {"TTA" in best_row["Modelo / Estratégia"]}')
print('=' * 70)
print(f'  Δ QWK vs melhor histórico (0.8754): {best_row["QWK"]-0.8754:+.4f}')
print('=' * 70)

## 17. Ensemble dos 5 Mais Dissimilares (Double Fault)

Seleciona o subconjunto de 5 modelos com **mínima dupla falha total** (máxima diversidade).

Com 6 modelos, existem C(6,5)=6 subsets possíveis — busca exaustiva, sem heurística.

**Critério:** minimizar Σ DF(i,j) sobre todos os pares do subconjunto.


In [ ]:
from itertools import combinations

# ── Double Fault matrix (validação) ───────────────────────────────────
val_preds_dict = {
    'B0':      val_preds_b0,
    'V2S':     val_preds_v2s,
    'B3':      val_preds_b3,
    'SWA':     val_preds_swa,
    'MIL-B0':  val_preds_mil,
    'SwinMIL': val_preds_swmil,
}
N_val = len(val_targets)
M6    = len(model_names)   # 6

df_mat = np.zeros((M6, M6))
for i, ni in enumerate(model_names):
    for j, nj in enumerate(model_names):
        both_wrong = (val_preds_dict[ni] != val_targets) & (val_preds_dict[nj] != val_targets)
        df_mat[i, j] = both_wrong.sum() / N_val
np.fill_diagonal(df_mat, 0.0)

# ── Busca exaustiva: subconjunto de 5 com menor DF total ──────────────
best_subset = None
best_df_sum = float('inf')
for subset in combinations(range(M6), 5):
    total = sum(df_mat[i, j] for i, j in combinations(subset, 2))
    if total < best_df_sum:
        best_df_sum = total
        best_subset = subset

selected_idx   = list(best_subset)
dropped_idx    = [i for i in range(M6) if i not in selected_idx][0]
selected_names = [model_names[i] for i in selected_idx]
dropped_name   = model_names[dropped_idx]

# ── Relatório de seleção ───────────────────────────────────────────────
avg_df = df_mat.sum(axis=1) / (M6 - 1)
print('Average DF por modelo (val):')
for i, n in enumerate(model_names):
    mark = '  ← REMOVIDO' if i == dropped_idx else ''
    print(f'  {n:<10}  avg_DF={avg_df[i]:.4f}{mark}')

print(f'\nModelo removido  : {dropped_name}')
print(f'Ensemble selecionado (5): {selected_names}')
print(f'Total DF do subset: {best_df_sum:.4f}  '
      f'(vs full 6: {sum(df_mat[i,j] for i,j in combinations(range(M6),2)):.4f})')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Heatmap DF completo com highlight do modelo removido ───────────────
ax = axes[0]
sns.heatmap(df_mat, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=model_names, yticklabels=model_names,
            ax=ax, mask=np.eye(M6, dtype=bool), linewidths=0.5, vmin=0)
ax.set_title('Matriz Double Fault (validação)\n(off-diagonal)', fontsize=11)
ax.tick_params(axis='x', rotation=30)
# Highlight dropped row/col
for spine in ax.spines.values():
    spine.set_visible(False)
r = dropped_idx
ax.add_patch(plt.Rectangle((r, 0), 1, M6, fill=False, edgecolor='red', lw=2.5))
ax.add_patch(plt.Rectangle((0, r), M6, 1, fill=False, edgecolor='red', lw=2.5))

# ── Bar chart: avg DF por modelo ──────────────────────────────────────
ax = axes[1]
colors = ['#e74c3c' if i == dropped_idx else '#3498db'
          for i in range(M6)]
bars = ax.bar(model_names, avg_df, color=colors, edgecolor='white')
ax.set_ylabel('Average DF com demais modelos', fontsize=10)
ax.set_title('Average DF por Modelo\nVermelho = removido (mais redundante)', fontsize=11)
ax.grid(axis='y', alpha=0.35)
for bar, v in zip(bars, avg_df):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.0005,
            f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('logs/df5-selection.png', dpi=200, bbox_inches='tight')
plt.show()


### 17b. Avaliação do Ensemble de 5 Modelos no Conjunto de Teste

In [ ]:
# ── Mapas de probs do conjunto de teste para os 5 selecionados ─────────
test_probs_map = {
    'B0':      test_probs_b0,
    'V2S':     test_probs_v2s,
    'B3':      test_probs_b3,
    'SWA':     test_probs_swa,
    'MIL-B0':  test_probs_mil,
    'SwinMIL': test_probs_swmil,
}
val_probs_map = {
    'B0':      val_probs_b0,
    'V2S':     val_probs_v2s,
    'B3':      val_probs_b3,
    'SWA':     val_probs_swa,
    'MIL-B0':  val_probs_mil,
    'SwinMIL': val_probs_swmil,
}

probs5_test = [test_probs_map[n] for n in selected_names]
probs5_val  = [val_probs_map[n]  for n in selected_names]

# QWK-calibrated weights para os 5 selecionados
kappas5 = np.array([
    cohen_kappa_score(val_targets, decode_ordinal(val_probs_map[n]),
                      weights='quadratic')
    for n in selected_names
])
qwk_weights5 = kappas5 / kappas5.sum()

print('QWK val e pesos (5 modelos):')
for n, k, w in zip(selected_names, kappas5, qwk_weights5):
    print(f'  {n:<10}  QWK_val={k:.4f}  peso={w:.4f}')

# Estratégias sobre os 5
results_5 = {}
strategies_5 = ensemble_strategies(probs5_test, weights=qwk_weights5.tolist())
for strat_name, preds in strategies_5.items():
    results_5[f'DF5-{strat_name}'] = compute_metrics_bootstrap(
        test_targets, preds, N_BOOT)

# Alpha search nos 5 (CNN vs MIL — se houver ambos)
cnn_in_5 = [n for n in selected_names if n in ['B0','V2S','B3','SWA']]
mil_in_5  = [n for n in selected_names if n in ['MIL-B0','SwinMIL']]

if cnn_in_5 and mil_in_5:
    val_cnn5 = np.mean([val_probs_map[n] for n in cnn_in_5], axis=0)
    val_mil5 = np.mean([val_probs_map[n] for n in mil_in_5],  axis=0)
    alphas   = np.arange(0.0, 1.05, 0.05)
    a_kappas = []
    for alpha in alphas:
        mixed = alpha * val_mil5 + (1 - alpha) * val_cnn5
        a_kappas.append(cohen_kappa_score(val_targets, decode_ordinal(mixed),
                                          weights='quadratic'))
    best_alpha5 = float(alphas[np.argmax(a_kappas)])

    test_cnn5 = np.mean([test_probs_map[n] for n in cnn_in_5], axis=0)
    test_mil5 = np.mean([test_probs_map[n] for n in mil_in_5],  axis=0)
    preds_alpha5 = decode_ordinal(best_alpha5 * test_mil5 + (1 - best_alpha5) * test_cnn5)
    results_5[f'DF5-Alpha-{best_alpha5:.2f}'] = compute_metrics_bootstrap(
        test_targets, preds_alpha5, N_BOOT)
    print(f'\nAlpha ótimo MIL/CNN (5 modelos, val): {best_alpha5:.2f}')

print(f'\n{len(results_5)} estratégias avaliadas.')


In [ ]:
# ── Tabela comparativa: 5 vs 6 modelos ────────────────────────────────
rows = []
# Melhores resultados do ensemble completo (cell 24 / df_results já existe)
for name, m in all_results.items():
    rows.append({'config': f'6-models | {name}',
                 'QWK':  round(m['kappa']['mean'], 4),
                 'Acc':  round(m['acc']['mean']*100, 2),
                 'F1':   round(m['f1']['mean'], 4)})

for name, m in results_5.items():
    rows.append({'config': name,
                 'QWK':  round(m['kappa']['mean'], 4),
                 'Acc':  round(m['acc']['mean']*100, 2),
                 'F1':   round(m['f1']['mean'], 4)})

df_compare = (pd.DataFrame(rows)
                .sort_values('QWK', ascending=False)
                .reset_index(drop=True))

# Destaca os 5 primeiros de cada grupo
print('TOP 5 — 6-model ensemble:')
six_rows = df_compare[df_compare['config'].str.startswith('6-models')]
print(six_rows.head(5).to_string(index=False))

print('\nTOP 5 — DF-5 ensemble:')
five_rows = df_compare[df_compare['config'].str.startswith('DF5')]
print(five_rows.head(5).to_string(index=False))

best_6 = six_rows.iloc[0]
best_5 = five_rows.iloc[0]
delta  = best_5['QWK'] - best_6['QWK']
print(f'\nMelhor 6-model: {best_6["config"].split("|")[1].strip():<30}  QWK={best_6["QWK"]:.4f}')
print(f'Melhor DF-5   : {best_5["config"]:<30}  QWK={best_5["QWK"]:.4f}')
print(f'Delta QWK (DF5 vs 6): {delta:+.4f}')


In [ ]:
# ── Gráfico de barras horizontal: Top-15 estratégias (5 + 6 modelos) ──
top_n = 15
top_df = df_compare.head(top_n)

colors_bar = ['#e74c3c' if 'DF5' in r else '#3498db'
              for r in top_df['config']]
labels_bar = [r.replace('6-models | ', '') for r in top_df['config']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels_bar[::-1], top_df['QWK'][::-1],
               color=colors_bar[::-1], edgecolor='white', height=0.7)
ax.set_xlabel('Quadratic Weighted Kappa (QWK)', fontsize=11)
ax.set_title(f'Top {top_n} estratégias de ensemble\n'
             f'Azul = 6 modelos  |  Vermelho = DF-5 ({", ".join(selected_names)})',
             fontsize=11)
ax.axvline(best_6['QWK'], color='#3498db', linestyle='--', alpha=0.6,
           label=f'Melhor 6-model ({best_6["QWK"]:.4f})')
ax.axvline(best_5['QWK'], color='#e74c3c', linestyle='--', alpha=0.6,
           label=f'Melhor DF-5 ({best_5["QWK"]:.4f})')
for bar, v in zip(bars, top_df['QWK'][::-1].values):
    ax.text(v + 0.0003, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)
ax.legend(fontsize=9)
ax.set_xlim(top_df['QWK'].min() - 0.005, top_df['QWK'].max() + 0.01)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('logs/df5-vs-6-comparison.png', dpi=200, bbox_inches='tight')
plt.show()
